# Phase 4 (CJPE) - Longformer-MLA + LoRA / LoRMA+ + gated-attention pooling over 4096-token windows

**Attach as Kaggle inputs:** (1) the BPE dataset that holds `tokenizer.json` (+ `meta.json`), (2) the Longformer pretraining output (`best.pt` + `best_meta.json`),
(3) optional: the earlier Model-2 Phase-4 output - its tokenised train/dev/test cache is reused (saves the tokenisation time).

**Run order:** set `SMOKE = True` once (about 5-10 min, checks everything end to end), then `SMOKE = False`.
Set `TRAIN_BUDGET_MIN` to the GPU minutes you can still afford. `ADAPTER = "lora"` or `"lorma_plus"` (the compulsory LoRMA experiment).
A dead/limited session is resumed automatically: attach the previous output as an input and run again.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
import re, gc, sys, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
warnings.filterwarnings("ignore")
T0 = time.time()

# USER SETTINGS  (the only block you normally touch)
SMOKE            = False        # True = ~5 min dry run on a tiny subset. RUN THIS ONCE FIRST.
MODE             = "train_predict"   # "train_predict" | "predict_only" (needs best_adapter.pt from an earlier run)
TRAIN_BUDGET_MIN = 480          # total training minutes - SET THIS TO WHAT YOUR QUOTA ALLOWS (LR schedule is time-based, so it always finishes annealing)
MAX_EPOCHS       = 5            # hard cap on passes over the train docs
SESSION_LIMIT_H  = 11.0         # stop training in time to still run inference before Kaggle's 12h wall
RESERVE_MIN      = 90           # minutes kept free for dev + test inference (4096-token windows are ~4x dearer than Model-2's 1024)

# optional manual overrides (otherwise auto-found under /kaggle/input)
TOKENIZER_JSON = None           # e.g. "/kaggle/input/<bpe-dataset>/tokenizer.json"
CKPT_PATH      = None           # e.g. "/kaggle/input/<longformer-dataset>/best.pt"  (auto-found: best.pt whose tag contains 'longformer')

# data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {"train": "Phase_4_CJPE/Train/train.json",
                "dev":   "Phase_4_CJPE/Train/dev.json",
                "test":  "Phase_4_CJPE/Test/test.json"}
LOWERCASE = True                    # dev/test are lowercased, train is partly mixed-case -> match the test distribution
EXCLUDE_TEST_IDS_FROM_DEV = True    # all 1,517 test IDs also sit in dev.json; keep them out of model selection / threshold tuning
MAX_CHARS, HEAD_CHARS, TAIL_CHARS = 400_000, 80_000, 240_000   # only affects the ~0.2% monster documents

CHUNK  = 4096
STRIDE = 3584
K_TRAIN = 4               # windows per document per training step (head 1 + tail 2 + stratified middle) = up to 16k tokens / doc
K_QUICK = 4               # windows per doc during quick validation
MAX_INFER_WINDOWS = 14    # windows per doc at final inference: 14 x 3584 + 512 = ~50k tokens, i.e. nearly every document is covered completely
NB = 8                    # relative-position buckets (+ first + last = NB + 2 embeddings)

# adapter
ADAPTER = "lora"          # "lora" (additive) | "lorma_plus" (multiplicative, (I + s*BA) W0 - see notes)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ("q_down", "q_up", "kv_down", "k_up", "v_up", "o_proj", "w_gate", "w_up", "w_down")   # every MLA + SwiGLU projection (k_rope_proj is tiny, left frozen)
BACKBONE_DROPOUT = 0.1    # the Longformer was pretrained with 0.1
GRAD_CKPT = True          # recompute block activations in backward: ~-60% memory for ~+30% time (needed for 4096-token windows on a 16GB GPU)

# optimisation
MICRO_DOCS, ACCUM = 2, 8              # 2 docs x 4 windows = up to 8 x 4096 tokens / micro-batch; 16 docs / optimizer step
LR_LORA, LR_HEAD, WD_HEAD = 3e-4, 1e-3, 0.01
WARMUP_STEPS, MIN_LR_RATIO, GRAD_CLIP = 50, 0.05, 1.0
AUX_W = 0.3                           # weight of the per-window auxiliary loss (deep supervision for the MIL head)
LOG_EVERY, EVAL_EVERY_MIN, CKPT_EVERY_MIN = 20, 45.0, 15.0
N_QUICK = 200                         # dev docs used for the periodic quick validation
INFER_BS, DOC_BS = 8, 64              # windows per backbone batch (8 x 4096 tokens) / docs per head batch
SEED = 1234

if SMOKE:
    TRAIN_BUDGET_MIN, MAX_EPOCHS, RESERVE_MIN = 3, 1, 0
    EVAL_EVERY_MIN, CKPT_EVERY_MIN, N_QUICK, MAX_INFER_WINDOWS = 1.0, 1.0, 40, 8
    N_SMOKE = {"train": 160, "dev": 90, "test": 40}

# paths / device
IS_KAGGLE  = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK  = Path("/kaggle/working" if IS_KAGGLE else ".") / ("phase4_longformer_lora" + ("_smoke" if SMOKE else ""))
CACHE, CKPT, OUT, SUB = WORK / "cache", WORK / "ckpt", WORK / "out", WORK / "submission"
for _d in (CACHE, CKPT, OUT, SUB):
    _d.mkdir(parents=True, exist_ok=True)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP  = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16   # T4 / P100 have no bf16
OOM_ERR  = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)

def log(*a):
    print(*a, flush=True)

def fmt_t(sec):
    sec = int(max(sec, 0)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def find_files(name, roots):
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            for d, _, fs in os.walk(r):
                if name in fs: hits.append(os.path.join(d, name))
    return hits

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

log(f"device={DEVICE} | amp={USE_AMP} | SMOKE={SMOKE} | MODE={MODE} | work dir={WORK}")
if DEVICE.type == "cuda":
    log("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")      # Kaggle secret named HF
except Exception as e:
    print("no Kaggle secret 'HF' found (", type(e).__name__, ") - relying on the HF_TOKEN environment variable if it is set")
print("HF token available:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from tokenizers import Tokenizer

# ---------- locate tokenizer + Longformer checkpoint (attached Kaggle inputs) ----------
def locate_tokenizer():
    if TOKENIZER_JSON: return TOKENIZER_JSON
    hits = find_files("tokenizer.json", [INPUT_ROOT, str(WORK)])
    ranked = []
    for h in hits:
        meta = read_json(os.path.join(os.path.dirname(h), "meta.json")) or {}
        ranked.append((0 if meta.get("tokenizer_type") == "true_utf8_byte_level_bpe" else 1, h))
    if not ranked: raise FileNotFoundError("tokenizer.json not found - attach the BPE output dataset")
    return sorted(ranked)[0][1]

LF_TAG_KEY = "longformer"            # tag of the Longformer pretraining run: longformer_mla_swiglu_clean_bpe32k_ctx4096_v1

def locate_checkpoint():
    """best.pt of the Longformer-MLA pretraining run (lowest pretraining val loss); latest.pt only as a fallback.
    The Model-2 checkpoint has a tag that also contains 'clean_bpe32k', so we require 'longformer' in the tag."""
    if CKPT_PATH: return CKPT_PATH
    best = None
    for h in find_files("best.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "best_meta.json")) or {}
        if LF_TAG_KEY in str(meta.get("tag", LF_TAG_KEY)):
            vl = meta.get("val_loss", 1e9)
            if best is None or vl < best[0]: best = (vl, h)
    if best: log(f"Longformer best.pt val_loss={best[0]:.4f}"); return best[1]
    for h in find_files("latest.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "latest_meta.json")) or {}
        if LF_TAG_KEY in str(meta.get("tag", LF_TAG_KEY)):
            log("best.pt not found - falling back to latest.pt"); return h
    raise FileNotFoundError("Longformer best.pt / latest.pt not found - attach the Longformer pretraining output as a Kaggle input")

TOK_PATH  = locate_tokenizer()
MODEL_CKPT = locate_checkpoint()
tokenizer = Tokenizer.from_file(TOK_PATH)
tokenizer.no_truncation(); tokenizer.no_padding()
VOCAB = tokenizer.get_vocab_size()
assert VOCAB < 65536, "token ids are cached as uint16"
log(f"tokenizer : {TOK_PATH} (vocab {VOCAB})")
log(f"checkpoint: {MODEL_CKPT}")

# ---------- text preparation (mirrors the cleaning used for the pretraining corpus) ----------
_ZW   = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_HYPH = re.compile(r"(?<=\w)-\n(?=\w)")
_WRAP = re.compile(r"(?<=\w)\n(?=\w)")
_SP   = re.compile(r"[ \t]+")
_NLSP = re.compile(r"[ \t]*\n[ \t]*")
_NL3  = re.compile(r"\n{3,}")

def prep_text(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = _CTRL.sub(" ", _ZW.sub("", t)).replace("\ufffd", " ")
    t = _HYPH.sub("", t)
    t = _WRAP.sub(" ", t)                 # undo hard line-wraps (dev/test are wrapped at ~50 chars)
    t = _NL3.sub("\n\n", _NLSP.sub("\n", _SP.sub(" ", t))).strip()
    if len(t) > MAX_CHARS:                # keep the beginning and (mostly) the end of giant judgments
        t = t[:HEAD_CHARS] + "\n" + t[-TAIL_CHARS:]
    return t.lower() if LOWERCASE else t

class TokDS:
    """tokenised split stored as one flat uint16 array + offsets"""
    def __init__(self, ids, labels, flat, offs, has_upper):
        self.ids, self.labels, self.flat, self.offs, self.has_upper = list(ids), np.asarray(labels), flat, offs, has_upper
        self.n_tok = np.diff(offs)
    def __len__(self): return len(self.ids)
    def slice(self, i, s, l): return self.flat[self.offs[i] + s: self.offs[i] + s + l]
    def subset(self, idx):
        idx = np.asarray(idx)
        parts = [self.flat[self.offs[i]:self.offs[i + 1]] for i in idx]
        offs = np.concatenate([[0], np.cumsum([len(p) for p in parts])]).astype(np.int64)
        return TokDS([self.ids[i] for i in idx], self.labels[idx], np.concatenate(parts), offs, self.has_upper[idx])

SIG = f"{'smoke' if SMOKE else 'full'}_lc{int(LOWERCASE)}_ex{int(EXCLUDE_TEST_IDS_FROM_DEV)}_v{VOCAB}"

def _hf_token():
    t = os.environ.get("HF_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None

def _download(split):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE4_FILES[split], repo_type="dataset", token=_hf_token())

def _records(split):
    obj = read_json(_download(split))
    if isinstance(obj, dict):
        for k in ("data", "records", "examples"):
            if isinstance(obj.get(k), list): obj = obj[k]; break
    return obj

def _tokenise(texts, bs=128):
    flat, lens = [], []
    for i in range(0, len(texts), bs):
        for e in tokenizer.encode_batch(texts[i:i + bs], add_special_tokens=False):
            a = np.asarray(e.ids, dtype=np.uint16); flat.append(a); lens.append(len(a))
        if (i // bs) % 40 == 0: log(f"    tokenised {min(i + bs, len(texts)):,}/{len(texts):,}")
    return np.concatenate(flat), np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)

def build_split(split, records, with_labels):
    t0 = time.time()
    ids = [str(r["id"]) for r in records]
    raw = [r.get("text") or "" for r in records]
    has_upper = np.array([t[:5000] != t[:5000].lower() for t in raw])
    labels = np.array([int(r["label"]) for r in records]) if with_labels else np.full(len(records), -1)  # test labels are NEVER read
    flat, offs = _tokenise([prep_text(t) for t in raw])
    log(f"  {split}: {len(ids):,} docs, {len(flat)/1e6:.1f}M tokens in {(time.time()-t0)/60:.1f} min")
    return TokDS(ids, labels, flat, offs, has_upper)

def save_split(split, ds):
    np.savez(CACHE / f"{split}_{SIG}.npz", ids=np.array(ds.ids), labels=ds.labels, flat=ds.flat, offs=ds.offs, has_upper=ds.has_upper)

def load_split(split):
    p = CACHE / f"{split}_{SIG}.npz"
    if not p.exists():
        hits = find_files(p.name, [INPUT_ROOT])      # tokenised splits of an earlier Phase-4 run (same tokenizer + prep) are reused
        if not hits: return None
        p = Path(hits[0]); log(f"  reusing cached tokenised {split} split from {p}")
    z = np.load(p)
    return TokDS(z["ids"].tolist(), z["labels"], z["flat"], z["offs"], z["has_upper"])

def get_all_splits():
    cached = {s: load_split(s) for s in ("train", "dev", "test")}
    if all(v is not None for v in cached.values()):
        log("tokenised splits loaded from cache"); return cached
    rng = np.random.default_rng(SEED)
    log("downloading / reading CJPE json ...")
    test_rec = _records("test")
    test_ids = {str(r["id"]) for r in test_rec}              # IDs only - test labels are not touched
    test_rec = [{"id": r["id"], "text": r.get("text")} for r in test_rec]
    dev_rec = _records("dev")
    n_overlap = sum(str(r["id"]) in test_ids for r in dev_rec)
    log(f"!! dev/test ID overlap: {n_overlap:,} of {len(test_ids):,} test IDs are also in dev.json")
    if EXCLUDE_TEST_IDS_FROM_DEV:
        dev_rec = [r for r in dev_rec if str(r["id"]) not in test_ids]
        log(f"   -> dev reduced to {len(dev_rec):,} docs that are NOT test cases (used for selection + threshold tuning)")
    tr_rec = _records("train")
    if SMOKE:
        pick_ = lambda recs, n: [recs[i] for i in rng.choice(len(recs), min(n, len(recs)), replace=False)]
        tr_rec, dev_rec, test_rec = pick_(tr_rec, N_SMOKE["train"]), pick_(dev_rec, N_SMOKE["dev"]), pick_(test_rec, N_SMOKE["test"])
    out = {"train": build_split("train", tr_rec, True), "dev": build_split("dev", dev_rec, True), "test": build_split("test", test_rec, False)}
    del tr_rec, dev_rec, test_rec; gc.collect()
    for s, d in out.items(): save_split(s, d)
    return out

SPLITS = get_all_splits()
ds_train, ds_dev, ds_test = SPLITS["train"], SPLITS["dev"], SPLITS["test"]
assert len(set(ds_test.ids)) == len(ds_test.ids), "duplicate test ids"

# ---------- window geometry (pure numpy) ----------
def all_starts(n, W=CHUNK, stride=STRIDE):
    """start offsets of overlapping windows; the LAST window is anchored to the end of the document,
    so the tail of the judgment (reasoning + disposal) is always covered by a full-length window"""
    if n <= W: return np.array([0], dtype=np.int64)
    s = np.arange(0, n - W + 1, stride, dtype=np.int64)
    if s[-1] != n - W: s = np.append(s, n - W)
    return s

def pick(n_win, K, rng=None):
    """coverage-preserving selection of K of n_win windows: first 1 + last 2 always kept,
    the other K-3 are one-per-segment over the middle (random inside a segment when training, centred at eval)"""
    if n_win <= K: return np.arange(n_win)
    if K == 1: return np.array([n_win - 1])
    if K == 2: return np.array([0, n_win - 1])
    keep = {0, n_win - 2, n_win - 1}
    m = K - 3
    if m > 0:
        lo, hi = 1, n_win - 3
        edges = np.linspace(lo, hi + 1, m + 1)
        for i in range(m):
            a = int(np.ceil(edges[i])); b = max(int(np.ceil(edges[i + 1])), a + 1)
            keep.add(min(int(rng.integers(a, b)) if rng is not None else (a + b - 1) // 2, hi))
    return np.array(sorted(keep))

def bucket_ids(g, n_all):
    """position embedding id of a window: 0 = first, NB+1 = last, 1..NB = relative position in between"""
    g = np.asarray(g)
    if n_all <= 1: return np.zeros(len(g), dtype=np.int64)
    b = 1 + np.minimum((g / (n_all - 1) * NB).astype(np.int64), NB - 1)
    b = np.where(g == n_all - 1, NB + 1, b)
    return np.where(g == 0, 0, b).astype(np.int64)

# ---------- audit ----------
def audit(name, ds):
    nt = ds.n_tok; nw = np.array([len(all_starts(n)) for n in nt])
    log(f"{name:9s} docs={len(ds):6,} | tokens mean={nt.mean():7.0f} median={np.median(nt):6.0f} p90={np.percentile(nt,90):7.0f} "
        f"p99={np.percentile(nt,99):8.0f} | windows/doc mean={nw.mean():4.1f} p95={np.percentile(nw,95):3.0f} | "
        f"{ds.has_upper.mean()*100:3.0f}% mixed-case in source | {nw.sum():,} windows total")
    if (ds.labels >= 0).all():
        c = np.bincount(ds.labels, minlength=2)
        log(f"{'':9s} labels: class0={c[0]:,} ({100*c[0]/c.sum():.1f}%)  class1={c[1]:,} ({100*c[1]/c.sum():.1f}%)  imbalance={c.max()/max(c.min(),1):.2f}")
log("=" * 100)
audit("train", ds_train); audit("dev(clean)", ds_dev); audit("test", ds_test)
log("=" * 100)

cnt = np.bincount(ds_train.labels, minlength=2).astype(np.float64)
CLASS_W = torch.tensor(cnt.sum() / (2.0 * cnt), dtype=torch.float32, device=DEVICE)   # "balanced" inverse-frequency weights
log(f"class weights (train, inverse-frequency): {CLASS_W.tolist()}")

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 32_000
    d_model: int = 768
    n_layers: int = 10
    n_heads: int = 12
    qk_nope_dim: int = 64
    qk_rope_dim: int = 32
    v_head_dim: int = 64
    q_lora_rank: int = 384
    kv_lora_rank: int = 256
    ffn_hidden: int = 2048
    dropout: float = 0.1
    layer_windows: list = field(default_factory=list)       # local window per layer
    global_stride: int = 256
    max_len: int = 131_072
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(rot_dim, max_len, base=10000.0):
    # angles in float64 so that positions up to 131k stay precise, then stored as float32
    inv_freq = 1.0 / (base ** (torch.arange(0, rot_dim, 2, dtype=torch.float64) / rot_dim))
    freqs = torch.outer(torch.arange(max_len, dtype=torch.float64), inv_freq)
    return freqs.cos().float(), freqs.sin().float()          # each (max_len, rot_dim/2)


def apply_rope(x, cos, sin):
    # x: (B, H, T, rot_dim); cos/sin: (1, 1, T, rot_dim/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)


def sliding_global_attention(q, k, v, window, gidx, d_nope):
    """
    Causal sliding-window attention + strided global tokens, in O(T * (2*window + G)) memory/compute.

    q, k : (B, H, T, d_nope + d_rope)    v : (B, H, T, d_v)     gidx : (G,) sorted global positions (< T)

    LOCAL  : the sequence is cut in blocks of `window` tokens; block n sees block n-1 and block n, masked so every
             token sees exactly the previous `window` tokens (itself included).
    GLOBAL columns: every block also sees the global tokens that lie OUTSIDE its tokens' window (no double counting);
             their rotary part is zeroed -> content-only scores.
    GLOBAL rows   : the global tokens themselves attend to all earlier tokens, content-only; these rows replace
             the local result at the global positions.
    T may be any length (it is right-padded to a multiple of `window`; with causal masking the padding is never read).
    """
    B, H, T, dq = q.shape
    dv = v.size(-1)
    w = window
    G = gidx.numel()
    dev = q.device
    q0, k0, v0 = q, k, v
    pad = (-T) % w
    if pad:
        q, k, v = F.pad(q, (0, 0, 0, pad)), F.pad(k, (0, 0, 0, pad)), F.pad(v, (0, 0, 0, pad))
    nb = (T + pad) // w

    qb = q.reshape(B, H, nb, w, dq)
    kb = k.reshape(B, H, nb, w, dq)
    vb = v.reshape(B, H, nb, w, dv)
    k_prev = F.pad(kb[:, :, :-1], (0, 0, 0, 0, 1, 0))          # previous block (zeros before block 0)
    v_prev = F.pad(vb[:, :, :-1], (0, 0, 0, 0, 1, 0))
    kg = k0.index_select(2, gidx)                               # global keys, rotary part zeroed
    kg = torch.cat([kg[..., :d_nope], torch.zeros_like(kg[..., d_nope:])], dim=-1)
    vg = v0.index_select(2, gidx)
    k_all = torch.cat([kg[:, :, None].expand(B, H, nb, G, dq), k_prev, kb], dim=3)      # (B,H,nb,G+2w,dq)
    v_all = torch.cat([vg[:, :, None].expand(B, H, nb, G, dv), v_prev, vb], dim=3)
    L = G + 2 * w

    r = torch.arange(w, device=dev)
    blk = torch.arange(nb, device=dev)
    cur_ok = r[None, :] <= r[:, None]                                   # key c <= query r in the same block
    prev_ok = (blk > 0)[:, None, None] & (r[None, :] > r[:, None])[None]   # previous block: only the last w-1-r.. keys
    qpos = blk[:, None] * w + r[None, :]                                # (nb, w) absolute query positions
    glob_ok = gidx[None, None, :] <= (qpos[:, :, None] - w)             # global token older than the local window
    mask = torch.cat([glob_ok, prev_ok, cur_ok[None].expand(nb, w, w)], dim=-1)         # (nb, w, L)
    mask = mask[None].expand(B, nb, w, L).reshape(B * nb, 1, w, L)

    qf = qb.permute(0, 2, 1, 3, 4).reshape(B * nb, H, w, dq)
    kf = k_all.permute(0, 2, 1, 3, 4).reshape(B * nb, H, L, dq)
    vf = v_all.permute(0, 2, 1, 3, 4).reshape(B * nb, H, L, dv)
    o = F.scaled_dot_product_attention(qf, kf, vf, attn_mask=mask)
    o = o.reshape(B, nb, H, w, dv).permute(0, 2, 1, 3, 4).reshape(B, H, nb * w, dv)[:, :, :T]

    qg = q0.index_select(2, gidx)[..., :d_nope] * math.sqrt(d_nope / dq)
    gmask = (torch.arange(T, device=dev)[None, :] <= gidx[:, None])[None, None]          # (1,1,G,T)
    og = F.scaled_dot_product_attention(qg, k0[..., :d_nope], v0, attn_mask=gmask)
    return o.index_copy(2, gidx, og)


def dense_reference_attention(q, k, v, window, gidx, d_nope):
    """slow O(T^2) implementation of exactly the same sparsity pattern; used only by the self-test"""
    B, H, T, dq = q.shape
    scale = dq ** -0.5
    s_full = torch.einsum("bhid,bhjd->bhij", q, k) * scale
    s_nope = torch.einsum("bhid,bhjd->bhij", q[..., :d_nope], k[..., :d_nope]) * scale
    i = torch.arange(T, device=q.device)[:, None]
    j = torch.arange(T, device=q.device)[None, :]
    is_g = torch.zeros(T, dtype=torch.bool, device=q.device)
    is_g[gidx] = True
    loc = (j <= i) & (i - j < window)
    gcol = is_g[None, :] & (j <= i - window)
    allowed = torch.where(is_g[:, None], j <= i, loc | gcol)
    use_nope = is_g[:, None] | (~loc)
    s = torch.where(use_nope, s_nope, s_full).masked_fill(~allowed, float("-inf"))
    return torch.matmul(s.softmax(-1), v)


class MLASparseAttention(nn.Module):
    """Multi-head Latent Attention projections + Longformer-style sliding-window/global attention pattern"""

    def __init__(self, cfg, window):
        super().__init__()
        self.h = cfg.n_heads
        self.dn, self.dr, self.dv = cfg.qk_nope_dim, cfg.qk_rope_dim, cfg.v_head_dim
        self.window = window
        # queries: d_model -> q_lora -> heads * (nope + rope)
        self.q_down = nn.Linear(cfg.d_model, cfg.q_lora_rank, bias=False)
        self.q_norm = RMSNorm(cfg.q_lora_rank)
        self.q_up = nn.Linear(cfg.q_lora_rank, self.h * (self.dn + self.dr), bias=False)
        # keys / values: d_model -> latent c_kv -> heads * (k_nope, v);  + one shared rotary key
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora_rank, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_lora_rank)
        self.k_up = nn.Linear(cfg.kv_lora_rank, self.h * self.dn, bias=False)
        self.v_up = nn.Linear(cfg.kv_lora_rank, self.h * self.dv, bias=False)
        self.k_rope_proj = nn.Linear(cfg.d_model, self.dr, bias=False)
        self.o_proj = nn.Linear(self.h * self.dv, cfg.d_model, bias=False)

    def project(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_up(self.q_norm(self.q_down(x))).view(B, T, self.h, self.dn + self.dr).transpose(1, 2)
        q = torch.cat([q[..., :self.dn], apply_rope(q[..., self.dn:], cos, sin)], dim=-1)
        ckv = self.kv_norm(self.kv_down(x))                                    # (B, T, kv_lora)  <- the MLA latent
        k_nope = self.k_up(ckv).view(B, T, self.h, self.dn).transpose(1, 2)
        v = self.v_up(ckv).view(B, T, self.h, self.dv).transpose(1, 2)
        k_rope = apply_rope(self.k_rope_proj(x)[:, None], cos, sin)            # (B, 1, T, dr) shared by all heads
        k = torch.cat([k_nope, k_rope.expand(B, self.h, T, self.dr)], dim=-1)
        return q, k, v

    def forward(self, x, cos, sin, gidx):
        B, T, _ = x.shape
        q, k, v = self.project(x, cos, sin)
        out = sliding_global_attention(q, k, v, self.window, gidx, self.dn)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.h * self.dv))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg, window):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = MLASparseAttention(cfg, window)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, gidx):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin, gidx))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class LongformerMLA_LM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.grad_ckpt = False
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection
        self.blocks = nn.ModuleList([Block(cfg, w) for w in cfg.layer_windows])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.qk_rope_dim, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)                    # damp residual-branch outputs (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def global_positions(self, T, device):
        return torch.arange(0, T, self.cfg.global_stride, device=device)

    def encode(self, idx):
        """idx (B, T) -> final hidden states (B, T, d_model) and the global positions (G,). Any T <= cfg.max_len."""
        B, T = idx.shape
        assert T <= self.cfg.max_len, f"sequence of {T} tokens exceeds max_len={self.cfg.max_len}"
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        gidx = self.global_positions(T, idx.device)
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = grad_checkpoint(blk, x, cos, sin, gidx, use_reentrant=False)
            else:
                x = blk(x, cos, sin, gidx)
        return self.final_norm(x), gidx

    def forward(self, idx):
        h, _ = self.encode(idx)
        return F.linear(h, self.embed.weight)                          # weight tying: logits = h @ E^T


# ---------------- adapters ----------------
class AdaptedLinear(nn.Module):
    """frozen base Linear + low-rank trainable update.
       mode 'lora'       : y = W0 x + s * B A x                        (additive, Hu et al.)
       mode 'lorma_plus' : y = (I + s * B A) W0 x = y0 + s * B A y0    (LoRMA+ = multiplicative update with additive rank inflation, Bihany et al. 2025, eq. 8: h = (I + a/r * BA) W0 x; B=0 at init, A random)
       B is zero-initialised in both modes so training starts exactly at the pretrained model."""
    def __init__(self, base, r, alpha, p, mode):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        a_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, a_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(p)
    def forward(self, x):
        y = self.base(x)
        z = x if self.mode == "lora" else y
        return y + F.linear(F.linear(self.drop(z), self.A), self.B) * self.scale

def apply_adapters(model):
    n = 0
    for blk in model.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                lin = getattr(parent, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(parent, name, AdaptedLinear(lin, LORA_R, LORA_ALPHA, LORA_DROPOUT, ADAPTER)); n += 1
    return n

def build_backbone():
    ck = torch.load(MODEL_CKPT, map_location="cpu", weights_only=False)
    cd = dict(ck.get("config", {}))
    if "layer_windows" not in cd:
        raise RuntimeError(f"{MODEL_CKPT} is not a Longformer-MLA checkpoint (its config has no 'layer_windows'). "
                           "Attach the Longformer pretraining output (best.pt) - not the Model-2/Model-3 one.")
    cd["dropout"] = BACKBONE_DROPOUT
    cfg = ModelConfig(**{k: cd[k] for k in ModelConfig.__dataclass_fields__ if k in cd})
    assert cfg.vocab_size == VOCAB, f"tokenizer vocab {VOCAB} != model vocab {cfg.vocab_size}"
    assert CHUNK <= cfg.max_len, "CHUNK exceeds the RoPE table of the checkpoint"
    model = LongformerMLA_LM(cfg)
    model.load_state_dict(ck["model"], strict=True)          # strict: fails loudly if the architecture copy differs
    if "val_loss" in ck: log(f"loaded Longformer-MLA weights (pretraining val loss {ck['val_loss']:.4f}, global step {ck.get('global_step')})")
    del ck
    for p in model.parameters(): p.requires_grad = False
    n_ad = apply_adapters(model)
    model.grad_ckpt = GRAD_CKPT
    return model.to(DEVICE), cfg, n_ad

model, MCFG, n_adapted = build_backbone()
FEAT_DIM = 3 * MCFG.d_model
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Longformer-MLA: {n_total - n_train:,} frozen params | {n_adapted} adapted Linear layers ({ADAPTER}, r={LORA_R}) -> {n_train:,} trainable ({100*n_train/n_total:.2f}%)")
log(f"  local windows per layer {MCFG.layer_windows} | global stride {MCFG.global_stride} | d_model {MCFG.d_model} | {MCFG.n_layers} layers | grad-ckpt {GRAD_CKPT}")

def backbone_features(model, ids, lens):
    """ids (N,T) right-padded, lens (N,). Attention is causal everywhere (local window, global rows, global columns), so padding
    at the END never influences real tokens -> no padding mask is needed.
    Per-window feature = [hidden state of the LAST real token ; mean of all real-token hidden states ; mean of the GLOBAL-token
    hidden states]. The global tokens (positions 0, S, 2S, ...) attend to the whole window prefix in every layer, so they are the
    model's long-range aggregators; the last token adds the freshest local context. (Same recipe as LongDocClassifier.)"""
    h, gidx = model.encode(ids)
    h = h.float()
    B, T, _ = h.shape
    ar = torch.arange(T, device=ids.device)[None]
    m = (ar < lens[:, None]).unsqueeze(-1).float()
    last = h[torch.arange(B, device=ids.device), lens - 1]
    mean = (h * m).sum(1) / lens[:, None].float()
    gm = (gidx[None, :] < lens[:, None]).float()                          # (B, G): global tokens that are real
    mean_g = (h[:, gidx] * gm.unsqueeze(-1)).sum(1) / gm.sum(1, keepdim=True).clamp(min=1.0)
    return torch.cat([last, mean, mean_g], dim=-1)

class DocHead(nn.Module):
    """window features -> doc logits. Gated-attention multiple-instance pooling (Ilse et al. 2018) over windows,
    + learned position embeddings (first / relative position / last) + explicit tail-window branch + per-window auxiliary classifier."""
    def __init__(self, in_dim, d=256, att=128, p=0.15):
        super().__init__()
        self.norm, self.proj = nn.LayerNorm(in_dim), nn.Linear(in_dim, d)
        self.pos = nn.Embedding(NB + 2, d)
        self.drop = nn.Dropout(p)
        self.att_v, self.att_u, self.att_w = nn.Linear(d, att), nn.Linear(d, att), nn.Linear(att, 1)
        self.win_cls = nn.Linear(d, 2)
        self.doc_cls = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, 2))
    def forward(self, feats, buckets, mask):
        h = F.gelu(self.proj(self.drop(self.norm(feats)))) + self.pos(buckets)
        h = self.drop(h)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)
        w = torch.softmax(a, dim=-1) * mask
        pooled = (w.unsqueeze(-1) * h).sum(1)
        last_idx = mask.long().sum(1) - 1
        h_last = h[torch.arange(h.size(0), device=h.device), last_idx]      # windows are ordered => last valid = tail window
        return self.doc_cls(torch.cat([pooled, h_last], dim=-1)), self.win_cls(h), w

head = DocHead(FEAT_DIM).to(DEVICE)
log(f"MIL head params: {sum(p.numel() for p in head.parameters()):,}")

@torch.no_grad()
def attention_selftest():
    """sparse sliding-window/global attention vs the O(T^2) dense reference, in fp32, through the ADAPTED projections
    (a few seconds; catches a PyTorch/SDPA incompatibility before any hours are spent)"""
    model.eval(); seen = set()
    for blk in model.blocks:
        a = blk.attn
        if a.window in seen: continue
        seen.add(a.window)
        T = 2 * a.window + 77                                              # deliberately not a multiple of the window
        x = torch.randn(2, T, MCFG.d_model, device=DEVICE)
        cos, sin = model.rope_cos[:T][None, None], model.rope_sin[:T][None, None]
        gidx = model.global_positions(T, DEVICE)
        q, k, v = a.project(x, cos, sin)
        fast = sliding_global_attention(q, k, v, a.window, gidx, a.dn)
        ref = dense_reference_attention(q, k, v, a.window, gidx, a.dn)
        err = (fast - ref).abs().max().item()
        if not err < 1e-3: raise RuntimeError(f"sparse attention != dense reference (window {a.window}): max diff {err:.3e}")
        log(f"  self-test: window {a.window:4d} T={T} G={gidx.numel()} sparse vs dense max diff {err:.2e}")
attention_selftest()

def amp_sanity():
    """autocast (fp16/bf16) vs fp32 on real windows - falls back to fp32 if they disagree"""
    global USE_AMP
    if not USE_AMP: return
    seqs = [ds_train.slice(i, 0, min(CHUNK, ds_train.n_tok[i])) for i in range(min(4, len(ds_train)))]
    T = max(len(s) for s in seqs); ids = np.zeros((len(seqs), T), dtype=np.int64); lens = np.array([len(s) for s in seqs])
    for i, s in enumerate(seqs): ids[i, :len(s)] = s
    ids, lens = torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE)
    model.eval()
    with torch.no_grad():
        f32 = backbone_features(model, ids, lens)
        with torch.autocast("cuda", dtype=AMP_DTYPE): f16 = backbone_features(model, ids, lens)
    cos_min = F.cosine_similarity(f32, f16.float(), dim=-1).min().item()
    ok = bool(torch.isfinite(f16).all()) and cos_min > 0.98
    log(f"AMP sanity ({str(AMP_DTYPE).split('.')[-1]}): min cosine(fp32, autocast)={cos_min:.5f} finite={bool(torch.isfinite(f16).all())} -> {'autocast ON' if ok else 'DISABLED (fp32)'}")
    USE_AMP = ok
amp_sanity()

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score, confusion_matrix

def pad_windows(seqs):
    T = max(len(s) for s in seqs)
    arr = np.zeros((len(seqs), T), dtype=np.int64)                   # pad id 0; irrelevant under causal attention
    for i, s in enumerate(seqs): arr[i, :len(s)] = s
    return arr, np.array([len(s) for s in seqs], dtype=np.int64)

def make_train_batch(ds, idxs, rng):
    seqs, doc_of, slot_of, rows, ys = [], [], [], [], []
    for b, di in enumerate(idxs):
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        sel = pick(nw, K_TRAIN, rng)                                 # random middle windows each epoch = augmentation for long docs
        for slot, g in enumerate(sel):
            s = int(starts[g]); seqs.append(ds.slice(di, s, min(CHUNK, n - s))); doc_of.append(b); slot_of.append(slot)
        rows.append(bucket_ids(sel, nw)); ys.append(int(ds.labels[di]))
    ids, lens = pad_windows(seqs)
    B, Km = len(idxs), max(len(r) for r in rows)
    bucket = np.zeros((B, Km), dtype=np.int64); mask = np.zeros((B, Km), dtype=bool)
    for i, r in enumerate(rows): bucket[i, :len(r)] = r; mask[i, :len(r)] = True
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return dict(ids=t(ids), lens=t(lens), doc_of=t(np.array(doc_of)), slot_of=t(np.array(slot_of)),
                bucket=t(bucket), mask=t(mask), y=t(np.array(ys, dtype=np.int64)))

@torch.no_grad()
def extract_features(ds, doc_indices, kcap, bs=None, tag=""):
    """run the (LoRA-adapted) backbone over the selected overlapping windows of every doc; returns window features
       + bookkeeping so any doc-level aggregator / section ablation can be evaluated later without touching the GPU again"""
    bs = bs or INFER_BS
    was_training = model.training
    model.eval(); head.eval()
    items, ptr, gidx, nall = [], [0], [], []
    for di in doc_indices:
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        for g in pick(nw, kcap):
            s = int(starts[g]); items.append((int(di), s, min(CHUNK, n - s))); gidx.append(int(g))
        ptr.append(len(items)); nall.append(nw)
    feats = np.zeros((len(items), FEAT_DIM), dtype=np.float32)
    t0 = time.time(); nb = (len(items) + bs - 1) // bs
    order = np.argsort([l for _, _, l in items], kind="stable")            # similar lengths share a batch -> far less padding at 4096 tokens
    for bi in range(nb):
        sel = order[bi * bs:(bi + 1) * bs]
        chunk = [items[j] for j in sel]
        ids, lens = pad_windows([ds.slice(di, s, l) for di, s, l in chunk])
        with amp_ctx():
            f = backbone_features(model, torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE))
        feats[sel] = f.float().cpu().numpy()
        if tag and (bi % 100 == 0 or bi == nb - 1):
            log(f"    [{tag}] batch {bi+1}/{nb} | {len(items)/max(time.time()-t0,1e-6):.0f} windows/s (avg)" if bi else f"    [{tag}] {len(items):,} windows from {len(doc_indices):,} docs")
    if was_training: model.train(); head.train()
    return feats, np.array(ptr), np.array(gidx), np.array(nall)

@torch.no_grad()
def doc_probs(feats, ptr, gidx, nall, keep=None, agg="att", collect_att=False):
    """P(class=1) per document from cached window features.
       keep(g, n_all)->bool mask restricts WHICH windows the head may see (section ablations); agg='winmean' = mean of window logits (no attention)"""
    head.eval()
    ND = len(ptr) - 1; probs = np.zeros(ND, dtype=np.float64); att = []
    for b0 in range(0, ND, DOC_BS):
        docs = list(range(b0, min(ND, b0 + DOC_BS))); rows = []
        for d in docs:
            sel = np.arange(ptr[d], ptr[d + 1])
            if keep is not None:
                m = np.asarray(keep(gidx[sel], int(nall[d])), dtype=bool)
                sel = sel[m] if m.any() else sel[-1:]
            rows.append(sel)
        Km = max(len(r) for r in rows)
        Fb = np.zeros((len(docs), Km, FEAT_DIM), dtype=np.float32); Bk = np.zeros((len(docs), Km), dtype=np.int64); Mk = np.zeros((len(docs), Km), dtype=bool)
        for i, (d, r) in enumerate(zip(docs, rows)):
            Fb[i, :len(r)] = feats[r]; Bk[i, :len(r)] = bucket_ids(gidx[r], int(nall[d])); Mk[i, :len(r)] = True
        Ft, Bt, Mt = (torch.from_numpy(a).to(DEVICE) for a in (Fb, Bk, Mk))
        logits, wlog, w = head(Ft, Bt, Mt)
        if agg == "winmean":
            logits = (wlog * Mt.unsqueeze(-1)).sum(1) / Mt.sum(1, keepdim=True)
        probs[b0:b0 + len(docs)] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        if collect_att:
            wn = w.cpu().numpy()
            for i, r in enumerate(rows): att.append((Bk[i, :len(r)], wn[i, :len(r)]))
    return (probs, att) if collect_att else probs

# ---------------- metrics ----------------
def macro_f1_np(y, pred):
    f = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f))

def tune_threshold(y, p, lo=0.20, hi=0.80, step=0.01, smooth=5):
    """threshold maximising Macro-F1; the F1 curve is smoothed so one lucky threshold on a small dev set is not chosen"""
    grid = np.arange(lo, hi + 1e-9, step)
    f = np.array([macro_f1_np(y, (p >= t).astype(int)) for t in grid])
    fs = np.convolve(np.pad(f, smooth // 2, mode="edge"), np.ones(smooth) / smooth, mode="valid")
    i = int(np.argmax(fs)); return float(grid[i]), float(f[i]), float(macro_f1_np(y, (p >= 0.5).astype(int)))

def full_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    return dict(thr=thr, macro_f1=Fm, macro_precision=P, macro_recall=R, accuracy=accuracy_score(y, pred), auc=auc,
                p0=pc[0], r0=rc[0], f1_0=fc[0], p1=pc[1], r1=rc[1], f1_1=fc[1], confusion=confusion_matrix(y, pred, labels=[0, 1]).tolist())

# fixed quick-validation subset of the (leak-free) dev set
_rs = np.random.default_rng(SEED + 1)
QUICK_IDX = np.sort(_rs.choice(len(ds_dev), min(N_QUICK, len(ds_dev)), replace=False))

def quick_eval():
    was_training = model.training
    f, ptr, g, na = extract_features(ds_dev, QUICK_IDX, K_QUICK)
    p = doc_probs(f, ptr, g, na); y = ds_dev.labels[QUICK_IDX]
    thr, f1_tuned, f1_05 = tune_threshold(y, p)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    if was_training: model.train(); head.train()
    return dict(auc=auc, f1_05=f1_05, f1_tuned=f1_tuned, thr=thr)

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": trainable, "lr": LR_LORA, "weight_decay": 0.0},
                         {"params": list(head.parameters()), "lr": LR_HEAD, "weight_decay": WD_HEAD}], betas=(0.9, 0.999))
BASE_LRS = [LR_LORA, LR_HEAD]
_USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16        # bf16 needs no loss scaling
try:    scaler = torch.amp.GradScaler("cuda", enabled=_USE_SCALER)
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=_USE_SCALER)

import shutil
S = dict(epoch=0, doc_ptr=0, step=0, train_sec=0.0, docs=0, wins=0, best_auc=-1.0, best_step=-1)
LATEST, BEST = CKPT / "latest_cjpe_lf.pt", CKPT / "best_adapter_lf.pt"
RUN_CFG = dict(backbone="longformer_mla", adapter=ADAPTER, r=LORA_R, alpha=LORA_ALPHA, chunk=CHUNK, stride=STRIDE, k_train=K_TRAIN, lowercase=LOWERCASE, feat_dim=FEAT_DIM, targets=list(LORA_TARGETS))

def adapter_state():
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_adapter_state(lora_sd, head_sd):
    own = dict(model.named_parameters()); miss = 0
    for n, v in lora_sd.items():
        if n in own: own[n].data.copy_(v.to(own[n].device))
        else: miss += 1
    head.load_state_dict(head_sd)
    if miss: log(f"warning: {miss} adapter tensors in the checkpoint did not match this model")

def save_latest():
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "state": dict(S), "cfg": RUN_CFG}, LATEST)

def save_best(res):
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "cfg": RUN_CFG, "quick": res, "step": S["step"]}, BEST)

def try_resume():
    if SMOKE: return False
    cands = [p for p in [str(LATEST)] + find_files("latest_cjpe_lf.pt", [INPUT_ROOT]) if os.path.exists(p)]
    best = None
    for p in cands:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if ck.get("cfg", {}).get("adapter") == ADAPTER and ck["cfg"].get("feat_dim") == FEAT_DIM:
                if best is None or ck["state"]["train_sec"] > best[1]["state"]["train_sec"]: best = (p, ck)
        except Exception as e:
            log(f"could not read {p}: {e}")
    if best is None: return False
    p, ck = best
    load_adapter_state(ck["lora"], ck["head"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); S.update(ck["state"])
    log(f"RESUMED from {p}: step {S['step']}, epoch {S['epoch']}, {S['train_sec']/60:.0f} min already trained, best quick AUC {S['best_auc']:.4f}")
    # carry the previous best adapter over (it may live in a read-only /kaggle/input dir)
    prev_best = os.path.join(os.path.dirname(p), "best_adapter_lf.pt")
    if not BEST.exists() and os.path.exists(prev_best): shutil.copy(prev_best, BEST)
    return True

def lr_mult(step, progress):
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

def micro_step(idxs, rng):
    b = make_train_batch(ds_train, idxs, rng)
    with amp_ctx():
        feats = backbone_features(model, b["ids"], b["lens"])
    feats = feats.float()
    B, Km = b["mask"].shape
    padded = torch.zeros(B, Km, feats.size(-1), device=feats.device).index_put((b["doc_of"], b["slot_of"]), feats)
    logits, wlog, _ = head(padded, b["bucket"], b["mask"])
    loss_doc = F.cross_entropy(logits, b["y"], weight=CLASS_W)                                   # class-imbalance: inverse-frequency weights
    wy = b["y"][:, None].expand(B, Km)[b["mask"]]
    loss_aux = F.cross_entropy(wlog[b["mask"]], wy, weight=CLASS_W, label_smoothing=0.1)         # every window sees the doc label (noisy -> smoothed)
    loss = loss_doc + AUX_W * loss_aux
    if not torch.isfinite(loss): return float("nan"), 0
    scaler.scale(loss / ACCUM).backward()
    return float(loss.item()), int(b["ids"].size(0))

def train():
    N = len(ds_train); budget = TRAIN_BUDGET_MIN * 60.0
    session_cap = max(0.0, (SESSION_LIMIT_H * 60 - RESERVE_MIN) * 60 - (time.time() - T0))
    t_sess, sec0 = time.time(), S["train_sec"]
    last_ckpt = last_eval = t_log = time.time(); w_log, losses, n_oom, stop, docs_sess = 0, [], 0, None, 0
    model.train(); head.train()
    log(f"training: budget {TRAIN_BUDGET_MIN} min total ({sec0/60:.0f} done) | this session may use {session_cap/60:.0f} min | "
        f"{MICRO_DOCS*ACCUM} docs/step x {K_TRAIN} windows | {N:,} train docs")
    while stop is None and S["epoch"] < MAX_EPOCHS:
        perm = np.random.default_rng(SEED + S["epoch"]).permutation(N)
        rng = np.random.default_rng(SEED * 7 + S["epoch"] * 1000 + S["step"])
        while S["doc_ptr"] < N:
            sess = time.time() - t_sess; elapsed = sec0 + sess; S["train_sec"] = elapsed
            progress = max(elapsed / budget, (S["epoch"] * N + S["doc_ptr"]) / (MAX_EPOCHS * N))
            if progress >= 1.0: stop = "budget reached"; break
            if sess >= session_cap: stop = "session limit reached - resume next session"; break
            m = lr_mult(S["step"], progress)
            for g, base in zip(opt.param_groups, BASE_LRS): g["lr"] = base * m
            opt.zero_grad(set_to_none=True); nmb = 0
            for _ in range(ACCUM):
                if S["doc_ptr"] >= N: break
                idxs = perm[S["doc_ptr"]: S["doc_ptr"] + MICRO_DOCS]; S["doc_ptr"] += len(idxs)
                try:
                    l, nw = micro_step(idxs, rng)
                except OOM_ERR:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    log(f"  CUDA OOM #{n_oom} - skipped a micro-batch (lower MICRO_DOCS or K_TRAIN if this repeats)")
                    if n_oom > 5: raise
                    continue
                if nw: losses.append(l); w_log += nw; S["docs"] += len(idxs); S["wins"] += nw; docs_sess += len(idxs); nmb += 1
            if nmb == 0: continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + list(head.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update(); S["step"] += 1
            if S["step"] % LOG_EVERY == 0:
                dt = time.time() - t_log; wps = w_log / max(dt, 1e-6)
                eta_epoch = (N - S["doc_ptr"]) / max(docs_sess / max(sess, 1e-6), 1e-6)
                log(f"ep {S['epoch']} step {S['step']:5d} | docs {S['doc_ptr']:6,}/{N:,} | loss {np.mean(losses[-LOG_EVERY*ACCUM:]):.4f} | "
                    f"lr x{m:.2f} | {wps:5.1f} win/s | trained {fmt_t(elapsed)}/{fmt_t(budget)} | epoch ETA {fmt_t(eta_epoch)}")
                t_log, w_log = time.time(), 0
            now = time.time()
            if now - last_eval >= EVAL_EVERY_MIN * 60:
                r = quick_eval(); last_eval = time.time()
                better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
                if better: S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
                log(f"  >> quick-dev ({len(QUICK_IDX)} docs, {K_QUICK} windows): AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | "
                    f"macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
                t_log = time.time()
            if now - last_ckpt >= CKPT_EVERY_MIN * 60: save_latest(); last_ckpt = now
        if stop is None: S["epoch"] += 1; S["doc_ptr"] = 0
    # final bookkeeping
    S["train_sec"] = sec0 + (time.time() - t_sess)
    r = quick_eval()
    better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
    if better or not BEST.exists(): S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
    log(f"  >> final quick-dev: AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
    save_latest()
    log(f"training stopped: {stop or 'epochs finished'} | steps {S['step']} | docs seen {S['docs']:,} | windows {S['wins']:,} | best quick AUC {S['best_auc']:.4f} @ step {S['best_step']}")

if MODE == "train_predict":
    try_resume()
    train()
else:
    log("MODE=predict_only: skipping training")

In [ ]:
# (dev = dev.json minus every test-case ID, so nothing here touches test labels)
# load the best adapter (by quick-dev AUC); fall back to whatever is currently in memory
_best = None
for p in [str(BEST)] + find_files("best_adapter_lf.pt", [INPUT_ROOT]):
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if ck["cfg"].get("adapter") == ADAPTER and ck["cfg"].get("feat_dim") == FEAT_DIM: _best = (p, ck); break
        except Exception as e:
            log(f"could not read {p}: {e}")
if _best:
    load_adapter_state(_best[1]["lora"], _best[1]["head"])
    log(f"evaluating best adapter from {_best[0]} (step {_best[1].get('step')}, quick AUC {_best[1]['quick']['auc']:.4f})")
else:
    assert MODE == "train_predict", "predict_only needs a best_adapter.pt (attach the training run's output as an input)"
    log("no best_adapter.pt found - using the weights currently in memory")

t0 = time.time()
dev_idx = np.arange(len(ds_dev))
f_dev, ptr_dev, g_dev, n_dev = extract_features(ds_dev, dev_idx, MAX_INFER_WINDOWS, tag="dev")
log(f"dev feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / "dev_window_feats.npz", feats=f_dev, ptr=ptr_dev, gidx=g_dev, nall=n_dev, y=ds_dev.labels)   # reusable by later aggregator experiments
y_dev = ds_dev.labels
p_dev = doc_probs(f_dev, ptr_dev, g_dev, n_dev)

THR, f1_tuned, f1_05 = tune_threshold(y_dev, p_dev)
if abs(THR - 0.5) > 0.15: log(f"warning: tuned threshold {THR:.2f} is far from 0.5 - check for train/dev shift before trusting it")
M05, MT = full_metrics(y_dev, p_dev, 0.5), full_metrics(y_dev, p_dev, THR)
log("\n" + "=" * 90 + f"\nDEV RESULTS ({len(y_dev):,} docs, class balance {np.bincount(y_dev, minlength=2).tolist()})\n" + "=" * 90)
for name, m in (("threshold 0.50", M05), (f"tuned thr {THR:.2f}", MT)):
    log(f"{name:15s} | Macro-F1 {m['macro_f1']:.4f} | Macro-P {m['macro_precision']:.4f} | Macro-R {m['macro_recall']:.4f} | Acc {m['accuracy']:.4f} | AUC {m['auc']:.4f}")
    log(f"{'':15s} | class0 P/R/F1 {m['p0']:.3f}/{m['r0']:.3f}/{m['f1_0']:.3f} | class1 P/R/F1 {m['p1']:.3f}/{m['r1']:.3f}/{m['f1_1']:.3f} | confusion [[TN,FP],[FN,TP]] = {m['confusion']}")

ABL = [("all windows (attention)",            None,                                              "att"),
       ("all windows (mean of window logits)", None,                                             "winmean"),
       (f"coverage subset K={K_TRAIN} (training policy)", lambda g, n: np.isin(g, pick(n, K_TRAIN)), "att"),
       ("first window only",                   lambda g, n: g == 0,                               "att"),
       ("last window only",                    lambda g, n: g == n - 1,                           "att"),
       ("last 2 windows",                      lambda g, n: g >= n - 2,                           "att"),
       ("first + last 2 windows",              lambda g, n: (g == 0) | (g >= n - 2),              "att"),
       ("first half of document",              lambda g, n: g <= (n - 1) / 2,                     "att"),
       ("second half of document",             lambda g, n: g >= (n - 1) / 2,                     "att")]
rows = []
for name, keep, agg in ABL:
    p = p_dev if (keep is None and agg == "att") else doc_probs(f_dev, ptr_dev, g_dev, n_dev, keep=keep, agg=agg)
    thr_, f_t, f_05 = tune_threshold(y_dev, p)
    rows.append(dict(setting=name, AUC=roc_auc_score(y_dev, p), MacroF1_at_0_5=f_05, MacroF1_tuned_thr=f_t, thr=thr_))
abl_df = pd.DataFrame(rows).round(4)
log("\nSECTION ABLATION (same trained model, restricted input; 'tuned thr' column is optimistic - tuned on dev itself)")
log(abl_df.to_string(index=False))
abl_df.to_csv(OUT / "ablation_sections.csv", index=False)

# ---------- where does the attention go? ----------
_, att = doc_probs(f_dev, ptr_dev, g_dev, n_dev, collect_att=True)
mass = np.zeros(NB + 2); cnt_ = np.zeros(NB + 2)
for b, w in att:
    for bi, wi in zip(b, w): mass[bi] += wi; cnt_[bi] += 1
labels_b = ["first"] + [f"rel {i/NB:.2f}-{(i+1)/NB:.2f}" for i in range(NB)] + ["last"]
att_df = pd.DataFrame({"window position": labels_b, "mean attention weight": mass / np.maximum(cnt_, 1), "share of total attention": mass / mass.sum(), "n windows": cnt_.astype(int)}).round(4)
log("\nATTENTION BY WINDOW POSITION (dev)"); log(att_df.to_string(index=False)); att_df.to_csv(OUT / "attention_by_position.csv", index=False)

# ---------- error analysis ----------
pred_dev = (p_dev >= THR).astype(int); ntok = ds_dev.n_tok
grp = np.where((pred_dev == 1) & (y_dev == 1), "TP", np.where((pred_dev == 0) & (y_dev == 0), "TN", np.where((pred_dev == 1) & (y_dev == 0), "FP", "FN")))
err_df = pd.DataFrame({"group": grp, "tokens": ntok, "windows": n_dev, "p1": p_dev, "mixed_case_src": ds_dev.has_upper})
summ = err_df.groupby("group").agg(n=("tokens", "size"), mean_tokens=("tokens", "mean"), median_tokens=("tokens", "median"), mean_windows=("windows", "mean"), mean_p1=("p1", "mean"), mixed_case_share=("mixed_case_src", "mean")).round(2)
log("\nERROR ANALYSIS at tuned threshold (FP = predicted 1, true 0 | FN = predicted 0, true 1)"); log(summ.to_string())
q = pd.qcut(ntok, 4, duplicates="drop"); er = pd.DataFrame({"len_quartile": q, "wrong": pred_dev != y_dev, "y": y_dev}).groupby("len_quartile", observed=True).agg(docs=("wrong", "size"), error_rate=("wrong", "mean"), class1_share=("y", "mean")).round(3)
log("\nERROR RATE BY DOCUMENT LENGTH QUARTILE (tokens)"); log(er.to_string())
summ.to_csv(OUT / "error_groups.csv"); er.to_csv(OUT / "error_by_length.csv")

pd.DataFrame({"id": ds_dev.ids, "label": y_dev, "p1": p_dev, "pred": pred_dev, "tokens": ntok, "windows": n_dev}).to_csv(OUT / "dev_probs.csv", index=False)
json.dump({"threshold": THR, "dev_at_0.5": M05, "dev_at_tuned": MT, "run": RUN_CFG, "train_state": S if "S" in globals() else None}, open(OUT / "dev_metrics.json", "w"), indent=1, default=float)

In [ ]:
t0 = time.time()
test_idx = np.arange(len(ds_test))
f_te, ptr_te, g_te, n_te = extract_features(ds_test, test_idx, MAX_INFER_WINDOWS, tag="test")
log(f"test feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / "test_window_feats.npz", feats=f_te, ptr=ptr_te, gidx=g_te, nall=n_te)
p_te = doc_probs(f_te, ptr_te, g_te, n_te)
pred_te = (p_te >= THR).astype(int)

pd.DataFrame({"id": ds_test.ids, "p1": p_te, "pred": pred_te}).to_csv(OUT / "test_probs.csv", index=False)   # probabilities kept for later ensembling
submission = [{"id": i, "pred": int(v)} for i, v in zip(ds_test.ids, pred_te)]

# ---- validation of the required format ----
assert len(submission) == len(ds_test) == len(set(ds_test.ids)), "one prediction per test case required"
assert all(set(s) == {"id", "pred"} and type(s["pred"]) is int and s["pred"] in (0, 1) for s in submission)
if not SMOKE:
    ref = [str(r["id"]) for r in _records("test")]
    assert [s["id"] for s in submission] == ref, "submission ids/order differ from test.json"
sub_path = SUB / "prediction.json"
with open(sub_path, "w", encoding="utf-8") as f: json.dump(submission, f)
import zipfile
with zipfile.ZipFile(SUB / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_path, "prediction.json")

log("\n" + "=" * 90)
log(f"SUBMISSION WRITTEN: {sub_path}   ({len(submission):,} predictions{' - SMOKE RUN, NOT A VALID SUBMISSION' if SMOKE else ''})")
log(f"predicted class balance: class0={int((pred_te==0).sum()):,} ({100*(pred_te==0).mean():.1f}%) | class1={int((pred_te==1).sum()):,} ({100*(pred_te==1).mean():.1f}%) | threshold {THR:.2f}")
log(f"dev Macro-F1 (leak-free dev, tuned thr): {MT['macro_f1']:.4f} | at 0.5: {M05['macro_f1']:.4f} | AUC {MT['auc']:.4f}")
log(f"total wall time: {(time.time()-T0)/60:.0f} min | outputs in {WORK}")
log(f"model cost: {n_total:,} params ({n_train:,} trainable adapter params, {FEAT_DIM}-d window features) | 4096-token windows, sliding-window + global attention (O(T*(w+T/S))), MLA latent KV {MCFG.kv_lora_rank}+{MCFG.qk_rope_dim} per token per layer")
if DEVICE.type == "cuda": log(f"peak GPU memory allocated: {torch.cuda.max_memory_allocated()/2**30:.1f} GB")
log("=" * 90)